In [2]:
import os
import random
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np


@dataclass
class Config:
    # Input/output
    bg_dir: str = r"D:\dataset_pre\empty"
    out_dir: str = r"D:\dataset_pre\fake"

    # How many images to generate
    num_images: int = 1

    # Rotation options (90-degree multiples)
    rotations = (0, 90, 180, 270)

    # Dish area (assume centered)
    dish_radius_ratio: float = 0.46  # relative to min(image width, height)
    inner_radius_ratio: float = 0.82 # keep colonies closer to center
    dish_edge_margin: int = 20       # extra safety margin inside the dish

    # Colony size (diameter) in pixels
    min_d: int = 35
    max_d: int = 50
    mean_d: int = 42
    std_d: int = 8

    # Colonies per image
    min_colonies: int = 30
    max_colonies: int = 100

    # Non-overlap margin
    min_gap: int = 4

    # Color (BGR) base for "milky white with slight yellow"
    # Averaged from samples: C2B69E / CFC3AB / B8AC96 / B9AD96
    base_bgr: tuple = (157, 180, 192)
    lab_jitter_l: int = 0
    lab_jitter_a: int = 0   # keep a* very tight to avoid green/red
    lab_jitter_b: int = 4   # allow slight yellow/blue variation

    # Texture/noise
    per_colony_noise: int = 2
    blur_ksize: int = 9  # must be odd
    blur_sigma: float = 3.0
    alpha_min: float = 0.85
    alpha_max: float = 0.95

    # Shape irregularity
    edge_jitter_ratio: float = 0.01  # subtle radius noise
    edge_points: int = 36
    inner_highlight_ratio: float = 0.6
    inner_lighten_l: int = 4

    # Random seed (set to int for reproducibility)
    seed: int | None = None


def _list_images(folder):
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    return [p for p in Path(folder).iterdir() if p.suffix.lower() in exts]


def _rotate_image(img, deg):
    if deg == 0:
        return img
    if deg == 90:
        return cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    if deg == 180:
        return cv2.rotate(img, cv2.ROTATE_180)
    if deg == 270:
        return cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)
    return img


def _sample_diameter(cfg: Config):
    d = int(np.clip(np.random.normal(cfg.mean_d, cfg.std_d), cfg.min_d, cfg.max_d))
    return d


def _random_color(cfg: Config):
    base = np.array([[cfg.base_bgr]], dtype=np.uint8)
    lab = cv2.cvtColor(base, cv2.COLOR_BGR2LAB).astype(np.int16)
    lab[0, 0, 0] += np.random.randint(-cfg.lab_jitter_l, cfg.lab_jitter_l + 1)
    lab[0, 0, 1] += np.random.randint(-cfg.lab_jitter_a, cfg.lab_jitter_a + 1)
    lab[0, 0, 2] += np.random.randint(-cfg.lab_jitter_b, cfg.lab_jitter_b + 1)
    lab = np.clip(lab, 0, 255).astype(np.uint8)
    bgr = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)[0, 0]
    return tuple(int(x) for x in bgr)


def _random_light_color(cfg: Config, base_bgr):
    base = np.array([[base_bgr]], dtype=np.uint8)
    lab = cv2.cvtColor(base, cv2.COLOR_BGR2LAB).astype(np.int16)
    lab[0, 0, 0] = np.clip(lab[0, 0, 0] + cfg.inner_lighten_l, 0, 255)
    lab = lab.astype(np.uint8)
    bgr = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)[0, 0]
    return tuple(int(x) for x in bgr)


def _inside_dish(cx, cy, r, center, dish_r, margin, inner_ratio):
    dx = cx - center[0]
    dy = cy - center[1]
    dist = (dx * dx + dy * dy) ** 0.5
    return dist + r + margin <= dish_r * inner_ratio


def _non_overlap(cx, cy, r, existing, min_gap):
    for (x, y, rr) in existing:
        dx = cx - x
        dy = cy - y
        if (dx * dx + dy * dy) ** 0.5 < (r + rr + min_gap):
            return False
    return True


def _draw_colony(img, cx, cy, r, cfg: Config):
    overlay = img.copy()
    color = _random_color(cfg)

    # Irregular edge mask
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    angles = np.linspace(0, 2 * np.pi, cfg.edge_points, endpoint=False)
    jitter = np.random.normal(0, cfg.edge_jitter_ratio, size=cfg.edge_points)
    radii = np.clip(r * (1.0 + jitter), r * 0.85, r * 1.15)
    pts = np.stack(
        [cx + radii * np.cos(angles), cy + radii * np.sin(angles)], axis=1
    ).astype(np.int32)
    cv2.fillPoly(mask, [pts], 255, lineType=cv2.LINE_AA)

    overlay[mask > 0] = color

    # inner highlight for slight "height" feeling
    inner_r = max(2, int(r * cfg.inner_highlight_ratio))
    light_color = _random_light_color(cfg, color)
    cv2.circle(overlay, (cx, cy), inner_r, light_color, thickness=-1, lineType=cv2.LINE_AA)

    # add localized noise
    noise = np.random.randint(-cfg.per_colony_noise, cfg.per_colony_noise + 1, img.shape, dtype=np.int16)
    noisy_overlay = np.clip(overlay.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    # soften edges with blurred mask
    if cfg.blur_ksize >= 3 and cfg.blur_ksize % 2 == 1:
        mask_f = cv2.GaussianBlur(mask, (cfg.blur_ksize, cfg.blur_ksize), cfg.blur_sigma)
    else:
        mask_f = mask

    alpha = np.random.uniform(cfg.alpha_min, cfg.alpha_max)
    alpha_mask = (mask_f.astype(np.float32) / 255.0) * alpha

    # blend only in masked area
    for c in range(3):
        img[..., c] = (
            img[..., c] * (1.0 - alpha_mask) + noisy_overlay[..., c] * alpha_mask
        ).astype(np.uint8)


def generate():
    cfg = Config()
    if cfg.seed is not None:
        random.seed(cfg.seed)
        np.random.seed(cfg.seed)

    bg_paths = _list_images(cfg.bg_dir)
    if not bg_paths:
        raise FileNotFoundError(f"No background images found in {cfg.bg_dir}")

    os.makedirs(cfg.out_dir, exist_ok=True)

    for idx in range(cfg.num_images):
        bg_path = random.choice(bg_paths)
        img = cv2.imread(str(bg_path))
        if img is None:
            continue

        rot = random.choice(cfg.rotations)
        img = _rotate_image(img, rot)

        h, w = img.shape[:2]
        center = (w // 2, h // 2)
        dish_r = int(min(w, h) * cfg.dish_radius_ratio)

        existing = []
        target_n = random.randint(cfg.min_colonies, cfg.max_colonies)
        attempts = 0
        max_attempts = target_n * 20

        while len(existing) < target_n and attempts < max_attempts:
            attempts += 1
            d = _sample_diameter(cfg)
            r = d // 2

            # sample within bounding square first
            cx = random.randint(r, w - r - 1)
            cy = random.randint(r, h - r - 1)

            if not _inside_dish(cx, cy, r, center, dish_r, cfg.dish_edge_margin, cfg.inner_radius_ratio):
                continue
            if not _non_overlap(cx, cy, r, existing, cfg.min_gap):
                continue

            _draw_colony(img, cx, cy, r, cfg)
            existing.append((cx, cy, r))

        out_path = Path(cfg.out_dir) / f"synthetic_{idx:05d}.jpg"
        cv2.imwrite(str(out_path), img)


if __name__ == "__main__":
    generate()


In [3]:
import json
import os
import random
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np


@dataclass
class Config:
    # Input/output
    bg_dir: str = r"D:\dataset_pre\empty"
    out_dir: str = r"D:\dataset_pre\fake"
    coco_in: str = r"E:\code\training code\instances_Train.json"
    coco_out: str = r"E:\code\training code\instances_Train_synthetic.json"
    append_to_input: bool = True
    category_name: str = "Arthrobacter oryzae"

    # How many images to generate
    num_images: int = 5

    # Rotation options (90-degree multiples)
    rotations = (0, 90, 180, 270)

    # Dish area (assume centered)
    dish_radius_ratio: float = 0.46  # relative to min(image width, height)
    inner_radius_ratio: float = 0.82 # keep colonies closer to center
    dish_edge_margin: int = 20       # extra safety margin inside the dish

    # Colony size (diameter) in pixels
    min_d: int = 35
    max_d: int = 50
    mean_d: int = 42
    std_d: int = 8

    # Colonies per image
    min_colonies: int = 30
    max_colonies: int = 100

    # Non-overlap margin
    min_gap: int = 4

    # Color (BGR) base for "milky white with slight yellow"
    # Averaged from samples: C2B69E / CFC3AB / B8AC96 / B9AD96
    base_bgr: tuple = (157, 180, 192)
    lab_jitter_l: int = 0
    lab_jitter_a: int = 0   # keep a* very tight to avoid green/red
    lab_jitter_b: int = 4   # allow slight yellow/blue variation

    # Texture/noise
    per_colony_noise: int = 2
    blur_ksize: int = 9  # must be odd
    blur_sigma: float = 3.0
    alpha_min: float = 0.85
    alpha_max: float = 0.95

    # Shape irregularity
    edge_jitter_ratio: float = 0.01  # subtle radius noise
    edge_points: int = 36
    inner_highlight_ratio: float = 0.6
    inner_lighten_l: int = 4

    # BBox loosen (COCO)
    bbox_expand_ratio_min: float = 0.02
    bbox_expand_ratio_max: float = 0.08
    bbox_shift_ratio_max: float = 0.03

    # Random seed (set to int for reproducibility)
    seed: int | None = None


def _list_images(folder):
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    return [p for p in Path(folder).iterdir() if p.suffix.lower() in exts]


def _rotate_image(img, deg):
    if deg == 0:
        return img
    if deg == 90:
        return cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    if deg == 180:
        return cv2.rotate(img, cv2.ROTATE_180)
    if deg == 270:
        return cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)
    return img


def _sample_diameter(cfg: Config):
    d = int(np.clip(np.random.normal(cfg.mean_d, cfg.std_d), cfg.min_d, cfg.max_d))
    return d


def _random_color(cfg: Config):
    base = np.array([[cfg.base_bgr]], dtype=np.uint8)
    lab = cv2.cvtColor(base, cv2.COLOR_BGR2LAB).astype(np.int16)
    lab[0, 0, 0] += np.random.randint(-cfg.lab_jitter_l, cfg.lab_jitter_l + 1)
    lab[0, 0, 1] += np.random.randint(-cfg.lab_jitter_a, cfg.lab_jitter_a + 1)
    lab[0, 0, 2] += np.random.randint(-cfg.lab_jitter_b, cfg.lab_jitter_b + 1)
    lab = np.clip(lab, 0, 255).astype(np.uint8)
    bgr = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)[0, 0]
    return tuple(int(x) for x in bgr)


def _random_light_color(cfg: Config, base_bgr):
    base = np.array([[base_bgr]], dtype=np.uint8)
    lab = cv2.cvtColor(base, cv2.COLOR_BGR2LAB).astype(np.int16)
    lab[0, 0, 0] = np.clip(lab[0, 0, 0] + cfg.inner_lighten_l, 0, 255)
    lab = lab.astype(np.uint8)
    bgr = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)[0, 0]
    return tuple(int(x) for x in bgr)


def _inside_dish(cx, cy, r, center, dish_r, margin, inner_ratio):
    dx = cx - center[0]
    dy = cy - center[1]
    dist = (dx * dx + dy * dy) ** 0.5
    return dist + r + margin <= dish_r * inner_ratio


def _non_overlap(cx, cy, r, existing, min_gap):
    for (x, y, rr) in existing:
        dx = cx - x
        dy = cy - y
        if (dx * dx + dy * dy) ** 0.5 < (r + rr + min_gap):
            return False
    return True


def _draw_colony(img, cx, cy, r, cfg: Config):
    overlay = img.copy()
    color = _random_color(cfg)

    # Irregular edge mask
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    angles = np.linspace(0, 2 * np.pi, cfg.edge_points, endpoint=False)
    jitter = np.random.normal(0, cfg.edge_jitter_ratio, size=cfg.edge_points)
    radii = np.clip(r * (1.0 + jitter), r * 0.85, r * 1.15)
    pts = np.stack(
        [cx + radii * np.cos(angles), cy + radii * np.sin(angles)], axis=1
    ).astype(np.int32)
    cv2.fillPoly(mask, [pts], 255, lineType=cv2.LINE_AA)

    overlay[mask > 0] = color

    # inner highlight for slight "height" feeling
    inner_r = max(2, int(r * cfg.inner_highlight_ratio))
    light_color = _random_light_color(cfg, color)
    cv2.circle(overlay, (cx, cy), inner_r, light_color, thickness=-1, lineType=cv2.LINE_AA)

    # add localized noise
    noise = np.random.randint(-cfg.per_colony_noise, cfg.per_colony_noise + 1, img.shape, dtype=np.int16)
    noisy_overlay = np.clip(overlay.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    # soften edges with blurred mask
    if cfg.blur_ksize >= 3 and cfg.blur_ksize % 2 == 1:
        mask_f = cv2.GaussianBlur(mask, (cfg.blur_ksize, cfg.blur_ksize), cfg.blur_sigma)
    else:
        mask_f = mask

    alpha = np.random.uniform(cfg.alpha_min, cfg.alpha_max)
    alpha_mask = (mask_f.astype(np.float32) / 255.0) * alpha

    # blend only in masked area
    for c in range(3):
        img[..., c] = (
            img[..., c] * (1.0 - alpha_mask) + noisy_overlay[..., c] * alpha_mask
        ).astype(np.uint8)


def _load_coco(path: str):
    if not path:
        return None
    p = Path(path)
    if not p.exists():
        return None
    with p.open("r", encoding="utf-8") as f:
        return json.load(f)


def _next_id(items, key="id", default_start=1):
    if not items:
        return default_start
    return max(int(it.get(key, 0)) for it in items) + 1


def _resolve_category_id(coco, cfg: Config):
    categories = coco.get("categories", []) if coco else []
    for cat in categories:
        if cat.get("name") == cfg.category_name:
            return int(cat.get("id", 1)), categories
    if categories:
        return int(categories[0].get("id", 1)), categories
    return 1, [{"id": 1, "name": cfg.category_name, "supercategory": "object"}]


def _loosen_bbox(cx, cy, r, w_img, h_img, cfg: Config):
    base_x0 = cx - r
    base_y0 = cy - r
    base_w = 2 * r
    base_h = 2 * r

    max_expand = cfg.bbox_expand_ratio_max
    min_expand = cfg.bbox_expand_ratio_min

    def rand_expand():
        return random.uniform(min_expand, max_expand)

    left = base_w * rand_expand()
    right = base_w * rand_expand()
    top = base_h * rand_expand()
    bottom = base_h * rand_expand()

    x0 = base_x0 - left
    y0 = base_y0 - top
    x1 = base_x0 + base_w + right
    y1 = base_y0 + base_h + bottom

    shift_x = base_w * random.uniform(-cfg.bbox_shift_ratio_max, cfg.bbox_shift_ratio_max)
    shift_y = base_h * random.uniform(-cfg.bbox_shift_ratio_max, cfg.bbox_shift_ratio_max)
    x0 += shift_x
    x1 += shift_x
    y0 += shift_y
    y1 += shift_y

    x0 = max(0.0, min(x0, w_img - 1.0))
    y0 = max(0.0, min(y0, h_img - 1.0))
    x1 = max(0.0, min(x1, w_img - 1.0))
    y1 = max(0.0, min(y1, h_img - 1.0))

    w = max(1.0, x1 - x0)
    h = max(1.0, y1 - y0)
    return [float(x0), float(y0), float(w), float(h)]


def generate():
    cfg = Config()
    if cfg.seed is not None:
        random.seed(cfg.seed)
        np.random.seed(cfg.seed)

    bg_paths = _list_images(cfg.bg_dir)
    if not bg_paths:
        raise FileNotFoundError(f"No background images found in {cfg.bg_dir}")

    os.makedirs(cfg.out_dir, exist_ok=True)

    coco_in = _load_coco(cfg.coco_in)
    if coco_in and cfg.append_to_input:
        coco_out = coco_in
    else:
        coco_out = {
            "info": coco_in.get("info", {}) if coco_in else {},
            "licenses": coco_in.get("licenses", []) if coco_in else [],
            "images": [],
            "annotations": [],
            "categories": [],
        }

    category_id, categories = _resolve_category_id(coco_out, cfg)
    coco_out["categories"] = categories

    image_id = _next_id(coco_out.get("images", []))
    ann_id = _next_id(coco_out.get("annotations", []))

    for idx in range(cfg.num_images):
        bg_path = random.choice(bg_paths)
        img = cv2.imread(str(bg_path))
        if img is None:
            continue

        rot = random.choice(cfg.rotations)
        img = _rotate_image(img, rot)

        h, w = img.shape[:2]
        center = (w // 2, h // 2)
        dish_r = int(min(w, h) * cfg.dish_radius_ratio)

        existing = []
        target_n = random.randint(cfg.min_colonies, cfg.max_colonies)
        attempts = 0
        max_attempts = target_n * 20

        while len(existing) < target_n and attempts < max_attempts:
            attempts += 1
            d = _sample_diameter(cfg)
            r = d // 2

            # sample within bounding square first
            cx = random.randint(r, w - r - 1)
            cy = random.randint(r, h - r - 1)

            if not _inside_dish(cx, cy, r, center, dish_r, cfg.dish_edge_margin, cfg.inner_radius_ratio):
                continue
            if not _non_overlap(cx, cy, r, existing, cfg.min_gap):
                continue

            _draw_colony(img, cx, cy, r, cfg)
            existing.append((cx, cy, r))

        out_path = Path(cfg.out_dir) / f"synthetic_{idx:05d}.jpg"
        cv2.imwrite(str(out_path), img)

        coco_out["images"].append(
            {
                "id": image_id,
                "file_name": out_path.name,
                "width": w,
                "height": h,
            }
        )

        for (cx, cy, r) in existing:
            bbox = _loosen_bbox(cx, cy, r, w, h, cfg)
            coco_out["annotations"].append(
                {
                    "id": ann_id,
                    "image_id": image_id,
                    "category_id": category_id,
                    "bbox": bbox,
                    "area": float(bbox[2] * bbox[3]),
                    "iscrowd": 0,
                    "segmentation": [],
                }
            )
            ann_id += 1

        image_id += 1

    with Path(cfg.coco_out).open("w", encoding="utf-8") as f:
        json.dump(coco_out, f, ensure_ascii=False, indent=2)


if __name__ == "__main__":
    generate()
